In [ ]:
#!/usr/bin/env python
# coding: utf-8

import pickle
import os
import numpy as np

from pyscf import gto
from pyscf.tools import cubegen

import py3Dmol

import H2O_hydroxide as Hh
from importlib import reload  # Python 3.4+


In [ ]:
# ============================================================
# Reconstruct the QM molecule
#
# This is the same QM geometry used for both QM/MM states.
# The MM water geometry changes, but the QM nuclear geometry
# does not.
# ============================================================

mol = gto.M(
    atom='''
        O    0.000    0.000    0.000
        H    0.970    0.000    0.000
    ''',
    basis='sto-3g',
    charge=-1,
    spin=0,
    unit='Angstrom',
    verbose=0
)

In [ ]:
# ============================================================
# Load saved QM/MM electronic states
# ============================================================

with open("qm_states.pkl", "rb") as f:
    qm_states = pickle.load(f)

print("Number of QM/MM states:", len(qm_states))

# State 0 = initial water geometry
# State 1 = optimized water geometry


states = []
mo_coeffs = []
mo_energies = []
mo_occs = []
homo_indices = []

iframe = 0
states.append(qm_states[iframe])
mo_coeffs.append(qm_states[iframe]["mo_coeff"])
mo_energies.append(qm_states[iframe]["mo_energy"])
mo_occs.append(qm_states[iframe]["mo_occ"])

iframe = 1
states.append(qm_states[iframe])
mo_coeffs.append(qm_states[iframe]["mo_coeff"])
mo_energies.append(qm_states[iframe]["mo_energy"])
mo_occs.append(qm_states[iframe]["mo_occ"])


In [ ]:
# ============================================================
# Basic orbital information for the final tate
# ============================================================
iframe = 1
nbasis, norbs = mo_coeffs[iframe].shape

print("For the final state ...")
print("Number of basis functions =", nbasis)
print("Number of orbitals        =", norbs)

print()
print("SCF energy =", states[iframe]["e_tot"], "Hartree")


# ============================================================
# Identify HOMO
# ============================================================

n_occ = mol.nelectron // 2
homo_index = n_occ - 1

print()
print("Number of occupied orbitals =", n_occ)
print("HOMO index                  =", homo_index)
print("HOMO energy                 =", mo_energies[iframe][homo_index], "Hartree")

In [ ]:
# ============================================================
# Generate orbital cube files
# ============================================================
resolution = 0.1
cube_folders = []
orbital_filenames = []
for iframe in range(2):
    cube_folder = "cubes_for_frame" + str(iframe)
    cube_folders.append(cube_folder)
    os.makedirs(cube_folder, exist_ok=True)
    orbital_filename = []
    for iorb in range(norbs):
    
        filename = f"{cube_folder}/orbital_{iorb}.cube"
        orbital_filename.append(filename)
        cubegen.orbital(
                mol,
                filename,
                mo_coeffs[iframe][:, iorb],
                resolution=resolution)
    orbital_filenames.append(orbital_filename)

iframe = 0; orbital = 5
print(cube_folders[iframe])
print(orbital_filenames[iframe][orbital])

In [ ]:
viewwidth = 500; viewheight = 500
for iframe in range(2):
    orbital_filename = orbital_filenames[iframe]; #print('looking at ', orbital_filename[iorbital])
    for iorbital in range(n_occ+1):
        view = py3Dmol.view(width=viewwidth,height=viewheight)
        Hh.show_orbital(view,orbital_filename,iorbital,mol)
        Hh.show_MM_atoms(view,"water_optimization.pkl",iframe)
        Hh.show_QM_atoms(view, mol)
        view.show()

# iorbital = 0
# iframe = 0
# view = py3Dmol.view(width=viewwidth,height=viewheight)
# orbital_filename = orbital_filenames[iframe]; #print('looking at ', orbital_filename[iorbital])
# Hh.show_orbital(view,orbital_filename,iorbital,mol)
# Hh.show_MM_atoms(view,"water_optimization.pkl",iframe)
# Hh.show_QM_atoms(view, mol)
# view.show()

# iorbital = 1
# iframe = 0
# view = py3Dmol.view(width=viewwidth,height=viewheight)
# orbital_filename = orbital_filenames[iframe]; #print('looking at ', orbital_filename[iorbital])
# Hh.show_orbital(view,orbital_filename,iorbital,mol)
# Hh.show_MM_atoms(view,"water_optimization.pkl",iframe)
# Hh.show_QM_atoms(view, mol)
# view.show()

In [ ]:
iwantdifferences = True
if iwantdifferences:

    Hh.cube_minmax('cubes_for_frame0/orbital_0.cube')
    Hh.cube_minmax('cubes_for_frame0/orbital_1.cube')
    
    # ============================================================
    # Generate sign-reversed cube for orbital 0
    # ============================================================
    
    iframe = 0
    iorbital = 1
    
    reversed_cube = f"cubes_for_frame{iframe}/orbital_{iorbital}_reversed.cube"
    
    cubegen.orbital(
        mol,
        reversed_cube,
        -mo_coeffs[iframe][:, iorbital],
        resolution=resolution
    )
    
    print("Wrote:", reversed_cube)
    
    view = py3Dmol.view(width=viewwidth,height=viewheight)
    orbital_filename = 'cubes_for_frame0/orbital_1_reversed.cube'; #print('looking at ', orbital_filename[iorbital])
    Hh.show_orbital(view,orbital_filename,iorbital,mol)
    Hh.show_MM_atoms(view,"water_optimization.pkl",iframe)
    Hh.show_QM_atoms(view, mol)
    view.show()

    cube_difference = "difference.cube"
    Hh.subtract_cube_files(
        orbital_filenames[1][iorbital],
        orbital_filenames[0][iorbital],
        cube_difference
    )
    
    view = py3Dmol.view(width=viewwidth,height=viewheight)
    Hh.show_orbital(view,cube_difference,0,mol,autoscale=True)
    Hh.show_MM_atoms(view,"water_optimization.pkl",iframe)
    Hh.show_QM_atoms(view, mol)
    view.show()

    cube_difference = "difference.cube"
    Hh.subtract_cube_files(
        orbital_filenames[0][iorbital],
        orbital_filenames[1][iorbital],
        cube_difference
    )
    
    view = py3Dmol.view(width=viewwidth,height=viewheight)
    Hh.show_orbital(view,cube_difference,0,mol,autoscale=True)
    Hh.show_MM_atoms(view,"water_optimization.pkl",iframe)
    Hh.show_QM_atoms(view, mol)
    view.show()
